# Phase 3 — Baseline OCR brut

**Objectif** : évaluer les performances d'un OCR appliqué **directement** sur les documents (sans aucun prétraitement), à la fois sur les documents originaux et sur les documents dégradés (Phase 2), afin de mesurer l'impact du bruit sur la reconnaissance de texte.

Moteur utilisé : **Tesseract OCR** (baseline standard, cf. taxonomie du cahier des charges).

Étapes :
1. Charger les manifests (documents originaux + documents dégradés de la Phase 2)
2. Appliquer l'OCR brut sur chaque image
3. Comparer avec la vérité terrain FUNSD (mots annotés)
4. Calculer CER, WER, taux de mots détectés/manquants
5. Analyser l'impact de chaque type et niveau de dégradation
6. Sauvegarder les résultats (tableau CSV + figures)

In [ ]:
import sys
import json
from pathlib import Path

sys.path.append("..")
from src.ocr_engine import run_ocr, ground_truth_text_from_annotation
from src.evaluation import evaluate_ocr_result

import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

ROOT = Path("..")
RAW_DIR = ROOT / "data" / "raw"
DEGRADED_DIR = ROOT / "data" / "degraded"
RESULTS_DIR = ROOT / "results"
(RESULTS_DIR / "tables").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "figures").mkdir(parents=True, exist_ok=True)

manifest = json.loads((RAW_DIR / "manifest.json").read_text(encoding="utf-8"))
manifest_degraded = json.loads((DEGRADED_DIR / "manifest_degraded.json").read_text(encoding="utf-8"))

print(f"Documents originaux : {len(manifest)}")
print(f"Images dégradées (Phase 2) : {len(manifest_degraded)}")

## 1. OCR baseline sur les documents ORIGINAUX (sans dégradation)

On applique Tesseract brut sur les 149 documents originaux et on calcule les métriques par rapport à la vérité terrain FUNSD.

In [ ]:
results_original = []

for entry in tqdm(manifest, desc="OCR sur documents originaux"):
    img_path = RAW_DIR / "images" / entry["image"]
    ann_path = RAW_DIR / "annotations" / entry["annotation"]

    gt_text = ground_truth_text_from_annotation(ann_path)
    ocr_result = run_ocr(img_path)
    metrics = evaluate_ocr_result(gt_text, ocr_result["text"])

    results_original.append({
        "document": entry["image"],
        "degradation": "aucune",
        "level": "original",
        "nb_mots_verite_terrain": len(gt_text.split()),
        "nb_mots_ocr": len(ocr_result["text"].split()),
        **metrics,
    })

df_original = pd.DataFrame(results_original)
df_original.to_csv(RESULTS_DIR / "tables" / "phase3_ocr_original.csv", index=False)
print(df_original[["cer", "wer", "taux_mots_detectes"]].describe())

## 2. OCR baseline sur les documents DÉGRADÉS (Phase 2)

Même traitement, appliqué à toutes les images dégradées générées en Phase 2 (8 dégradations × 3 niveaux × 149 documents). Cette cellule peut prendre du temps (plusieurs milliers d'images) — c'est normal, laisse tourner.

In [ ]:
results_degraded = []

for entry in tqdm(manifest_degraded, desc="OCR sur documents degrades"):
    img_path = DEGRADED_DIR / "images" / entry["image_degradee"]
    ann_path = RAW_DIR / "annotations" / entry["annotation_originale"]

    gt_text = ground_truth_text_from_annotation(ann_path)
    ocr_result = run_ocr(img_path)
    metrics = evaluate_ocr_result(gt_text, ocr_result["text"])

    results_degraded.append({
        "document": entry["document_original"],
        "degradation": entry["degradation"],
        "level": entry["level"],
        "nb_mots_verite_terrain": len(gt_text.split()),
        "nb_mots_ocr": len(ocr_result["text"].split()),
        **metrics,
    })

df_degraded = pd.DataFrame(results_degraded)
df_degraded.to_csv(RESULTS_DIR / "tables" / "phase3_ocr_degraded.csv", index=False)
print(f"{len(df_degraded)} résultats sauvegardés.")

## 3. Tableau récapitulatif : CER/WER moyens par dégradation et niveau

In [ ]:
df_all = pd.concat([df_original, df_degraded], ignore_index=True)
df_all.to_csv(RESULTS_DIR / "tables" / "phase3_ocr_all.csv", index=False)

summary = df_all.groupby(["degradation", "level"])[["cer", "wer", "taux_mots_detectes"]].mean().round(3)
summary = summary.sort_values("cer")
summary

## 4. Visualisation : impact de chaque dégradation sur le CER

In [ ]:
order = ["flou_gaussien","bruit_aleatoire","rotation_legere","faible_contraste",
         "compression_jpeg","effet_scan_degrade","distorsion_decalage","ombres"]
level_order = ["faible", "moyen", "fort"]

pivot_cer = df_degraded.groupby(["degradation","level"])["cer"].mean().unstack()[level_order].loc[order]

fig, ax = plt.subplots(figsize=(11,6))
pivot_cer.plot(kind="bar", ax=ax, color=["#9FD8CB","#1C7293","#21295C"])
baseline_cer = df_original["cer"].mean()
ax.axhline(baseline_cer, color="red", linestyle="--", label=f"CER original ({baseline_cer:.2f})")
ax.set_ylabel("CER (Character Error Rate)")
ax.set_xlabel("Type de dégradation")
ax.set_title("Impact des dégradations sur le CER — OCR brut (Tesseract)")
ax.legend(title="Niveau")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "phase3_cer_par_degradation.png", dpi=130)
plt.show()

## 5. Visualisation : impact de chaque dégradation sur le WER

In [ ]:
pivot_wer = df_degraded.groupby(["degradation","level"])["wer"].mean().unstack()[level_order].loc[order]

fig, ax = plt.subplots(figsize=(11,6))
pivot_wer.plot(kind="bar", ax=ax, color=["#9FD8CB","#1C7293","#21295C"])
baseline_wer = df_original["wer"].mean()
ax.axhline(baseline_wer, color="red", linestyle="--", label=f"WER original ({baseline_wer:.2f})")
ax.set_ylabel("WER (Word Error Rate)")
ax.set_xlabel("Type de dégradation")
ax.set_title("Impact des dégradations sur le WER — OCR brut (Tesseract)")
ax.legend(title="Niveau")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "phase3_wer_par_degradation.png", dpi=130)
plt.show()

## 6. Première analyse des types d'erreurs

Quelles dégradations sont les plus destructrices pour l'OCR, à niveau `fort` égal ?

In [ ]:
fort_only = df_degraded[df_degraded["level"] == "fort"].groupby("degradation")[["cer","wer","taux_mots_detectes"]].mean()
fort_only = fort_only.sort_values("cer", ascending=False)
print("Classement des dégradations les plus destructrices (niveau FORT), du pire au moins pire :")
fort_only

## 7. Note de synthèse (résultats attendus de la Phase 3)

- **Baseline OCR** : Tesseract appliqué brut, sans aucun prétraitement, sur 149 documents originaux + toutes les images dégradées de la Phase 2.
- **CER/WER de référence (documents propres)** : voir cellule 1 — sert de point de comparaison pour juger l'impact réel de chaque dégradation.
- **Impact des dégradations** : classement des 8 dégradations par sévérité (cellule 6) — permet d'identifier lesquelles cassent le plus l'OCR brut.
- **Tableaux sauvegardés** : `results/tables/phase3_ocr_original.csv`, `phase3_ocr_degraded.csv`, `phase3_ocr_all.csv`.
- **Figures sauvegardées** : `results/figures/phase3_cer_par_degradation.png`, `phase3_wer_par_degradation.png`.
- **Prochaine étape (Phase 4)** : appliquer différentes techniques de prétraitement OpenCV (binarisation, débruitage, correction d'inclinaison...) **avant** l'OCR, et comparer avec cette baseline pour voir quels prétraitements aident le plus, selon le type de dégradation.